In [12]:
import re
from pathlib import Path
from collections import defaultdict

def simplify_verilog_remove_port_names(input_path, output_path):
    with open(input_path, "r") as file:
        original_lines = file.readlines()

    # Preserve module line separately
    module_line = ""
    new_original_lines = []
    for line in original_lines:
        if line.strip().startswith("module "):
            module_line = line
        else:
            new_original_lines.append(line)

    # Remove the first 7 lines after module
    lines = new_original_lines[6:]

    buffer = ''
    inside_decl = False
    declared_lines = []
    new_lines = []

    # Group declaration and logic lines
    for line in lines:
        if re.match(r'^\s*(wire|input|output)', line) and ';' not in line:
            buffer = line
            inside_decl = True
        elif inside_decl:
            buffer += line
            if ';' in line:
                declared_lines.append(buffer)
                buffer = ''
                inside_decl = False
        elif not inside_decl and re.match(r'^\s*(wire|input|output).*;', line):
            declared_lines.append(line)
        else:
            new_lines.append(line)

    # Assign new names to all declared signals
    signal_map = {}
    cnt = 0
    renamed_lines = []

    for decl in declared_lines:
        match = re.match(r'^\s*(wire|input|output)(\s+\[.*?\])?\s+(.*);', decl.replace('\n', ' '))
        if not match:
            renamed_lines.append(decl)
            continue
        sig_type, bus, rest = match.groups()
        signals = [s.strip() for s in rest.split(',')]
        new_decls = []
        for sig in signals:
            original_sig = sig
            new_name = f"w_{cnt}"
            signal_map[original_sig] = new_name
            cnt += 1
            new_decls.append(new_name)
        renamed_lines.append((sig_type, bus if bus else '', new_decls))

    # Collapse declarations with same bus/type
    decl_by_type_bus = defaultdict(list)
    for sig_type, bus, names in renamed_lines:
        decl_by_type_bus[(sig_type, bus)].extend(names)

    collapsed_decls = []
    for (sig_type, bus), names in decl_by_type_bus.items():
        line = f"{sig_type} {bus} " if bus else f"{sig_type} "
        line += ', '.join(names) + ";"
        collapsed_decls.append(line)

    # Mapping gate types
    gate_type_map = {
        'DFFASRHQNx1_ASAP7_75t_R': 'dff',
        'AND2x2_ASAP7_75t_R': 'and',
        'OR2x2_ASAP7_75t_R': 'or',
        'INVx2_ASAP7_75t_R': 'not',
        'XOR2x1_ASAP7_75t_R': 'xor',
        'XNOR2x1_ASAP7_75t_R': 'xnor',
        'NOR2x1_ASAP7_75t_R': 'nor',
        'NAND2x1_ASAP7_75t_R': 'nand',
        'BUFx2_ASAP7_75t_R': 'buf'
    }
    dff_port_order = ['RN', 'SN', 'CK', 'D', 'Q']
    original_to_new = {
        'RESETN': 'RN',
        'SETN': 'SN',
        'CLK': 'CK',
        'D': 'D',
        'QN': 'Q'
    }

    gate_blocks = []
    buffer = ""
    in_gate = False

    # Extract gate blocks
    for line in new_lines:
        if re.match(r'^\s*\w+\s+\S+\s*\(.*', line):
            buffer = line
            in_gate = True
            if ');' in line:
                gate_blocks.append(buffer)
                buffer = ''
                in_gate = False
        elif in_gate:
            buffer += line
            if ');' in line:
                gate_blocks.append(buffer)
                buffer = ''
                in_gate = False
        else:
            gate_blocks.append(line)

    final_lines = []
    gate_cnt = 1
    default_const = "1'b1"

    for block in gate_blocks:
        if block.strip().startswith("module"):
            continue

        gate_match = re.match(r'^\s*(\w+)\s+(\S+)\s*\((.*)\);\s*$', block.replace('\n', ' ').strip())
        if gate_match:
            gate_type, gate_name, port_list = gate_match.groups()
            gate_type_simple = gate_type_map.get(gate_type, gate_type)
            gate_name = f"g_{gate_cnt}"
            gate_cnt += 1

            ports = port_list.split(',')
            port_dict = {}
            for port in ports:
                port = port.strip()
                p_match = re.match(r'\.(\w+)\((.*?)\)', port)
                if not p_match:
                    continue
                pname, sig = p_match.groups()
                sig = sig.strip()

                if sig.startswith('\\'):
                    lookup_key = sig
                else:
                    lookup_key = sig.split('[')[0] if '[' in sig else sig

                new_sig = signal_map.get(lookup_key, sig)
                if '[' in sig and not sig.startswith('\\'):
                    index = sig[sig.index('['):]
                    new_sig += index

                if gate_type_simple == 'dff' and pname in original_to_new:
                    port_dict[original_to_new[pname]] = new_sig
                else:
                    port_dict[pname] = new_sig

            # Format ports based on gate type
            if gate_type_simple == 'dff':
                new_ports = [f".{p}({port_dict.get(p, default_const)})" for p in dff_port_order]
            elif gate_type_simple in ['and', 'or', 'nor', 'nand', 'xor', 'xnor']:
                logic_order = ['Y', 'A', 'B']
                new_ports = [f"{port_dict.get(p, default_const)}" for p in logic_order]
            elif gate_type_simple in ['not', 'buf']:
                logic_order = ['Y', 'A']
                new_ports = [f"{port_dict.get(p, default_const)}" for p in logic_order]
            else:
                new_ports = [f".{k}({v})" for k, v in port_dict.items()]

            final_lines.append(f"{gate_type_simple} {gate_name} ( " + ', '.join(new_ports) + " );\n")
        else:
            final_lines.append(block)

    # Output final lines
    final_lines = [module_line] + collapsed_decls + final_lines

    with open(output_path, "w") as f:
        f.writelines(final_lines)

    print(f"Simplified file written to: {output_path}")


In [7]:
#simplify_verilog_remove_port_names ('./../Trojan1.vg', './simplified/Trojan1_simplified.v')
for i in range(0, 10):
    input_file = Path(f'./Trojans/Trojan{i}.vg')
    output_file = Path(f'./simplified_trojans/Trojan{i}_simplified.v')
    simplify_verilog_remove_port_names(input_file, output_file)

FileNotFoundError: [Errno 2] No such file or directory: 'Trojans/Trojan0.vg'

In [13]:
for i in range(0, 10):
    input_file = Path(f'./data/new_synthesized_Trojans/Trojan{i}.vg')
    output_file = Path(f'./data/new_standard_Trojans/Trojan{i}_standard.v')
    simplify_verilog_remove_port_names(input_file, output_file)

Simplified file written to: data/new_standard_Trojans/Trojan0_standard.v
Simplified file written to: data/new_standard_Trojans/Trojan1_standard.v
Simplified file written to: data/new_standard_Trojans/Trojan2_standard.v
Simplified file written to: data/new_standard_Trojans/Trojan3_standard.v
Simplified file written to: data/new_standard_Trojans/Trojan4_standard.v
Simplified file written to: data/new_standard_Trojans/Trojan5_standard.v
Simplified file written to: data/new_standard_Trojans/Trojan6_standard.v
Simplified file written to: data/new_standard_Trojans/Trojan7_standard.v
Simplified file written to: data/new_standard_Trojans/Trojan8_standard.v
Simplified file written to: data/new_standard_Trojans/Trojan9_standard.v


In [15]:
from exploit_gates import transform_and_parse_with_originals
for i in range(10):
    transform_and_parse_with_originals(f'./data/new_standard_Trojans/Trojan{i}_standard.v', f"./data/new_simplified_Trojans/simplified_Trojan{i}.v")